In [9]:
import pandas as pd
import numpy as np
from scipy.stats import f_oneway, kruskal
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

parquet_path = r"D:\\cmd3\\final_df.parquet"
df = pd.read_parquet(parquet_path)

print(df.shape)
df.head()

(1790, 77449)


,target4,study,age,PMID,number_reads,number_bases,minimum_read_length,median_read_length,BMI,hdl,...,pc__PWY.7234..inosine.5..phosphate.biosynthesis.III.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__PWY.6167..flavin.biosynthesis.II..archaea..g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__PWY.6876..isopropanol.biosynthesis.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__PWY.5101..L.isoleucine.biosynthesis.II.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__PWY.5189..tetrapyrrole.biosynthesis.II..from.glycine..g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__P261.PWY..coenzyme.M.biosynthesis.I.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__METHANOGENESIS.PWY..methanogenesis.from.H2.and.CO2.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__PWY.6349..CDP.archaeol.biosynthesis.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__PWY.5198..factor.420.biosynthesis.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus,pc__PWY.6350..archaetidylinositol.biosynthesis.g__Methanobrevibacter.s__Methanobrevibacter_arboriphilus
0,CRC,FengQ_2015,64.0,25758642.0,40898340.0,3.649611e+09,30.0,93.0,29.35,28.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,HC,FengQ_2015,68.0,25758642.0,66107961.0,6.196998e+09,30.0,96.0,32.00,50.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,HC,FengQ_2015,60.0,25758642.0,60789126.0,5.708593e+09,30.0,96.0,22.10,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,PA,FengQ_2015,70.0,25758642.0,50300253.0,4.741158e+09,30.0,96.0,34.11,74.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,HC,FengQ_2015,68.0,25758642.0,51945426.0,4.913627e+09,30.0,97.0,23.45,40.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
unique_target4 = sorted(df["target4"].dropna().unique().tolist())
print("Unique target4 values:")
for value in unique_target4:
    print(value)

print(f"\nTotal unique non-null values: {len(unique_target4)}")
print(f"Null count: {df['target4'].isna().sum()}")

Unique target4 values:
CRC
HC
Other
PA

Total unique non-null values: 4
Null count: 0


In [11]:
feature_cols = [col for col in df.columns if col not in ["target4", "study"]]
X_all = df[feature_cols].apply(pd.to_numeric, errors="coerce")
y_all = df["target4"].astype(str)
study_all = df["study"].astype(str)
n_classes = y_all.nunique()

def drop_duplicate_columns(frame):
    if frame.shape[1] == 0:
        return frame
    return frame.T.drop_duplicates().T

def filter_low_information_features(
    X_train_raw,
    X_test_raw,
    missing_thresh=0.4,
    zero_thresh=0.995,
    prevalence_thresh=0.01,
    variance_quantile=0.01,
):
    start_features = X_train_raw.shape[1]

    keep_non_na = X_train_raw.notna().any(axis=0)
    X_train = X_train_raw.loc[:, keep_non_na].copy()
    X_test = X_test_raw.loc[:, keep_non_na].copy()
    after_non_na = X_train.shape[1]

    missing_rate = X_train.isna().mean(axis=0)
    keep_missing = missing_rate <= missing_thresh
    X_train = X_train.loc[:, keep_missing].copy()
    X_test = X_test.loc[:, keep_missing].copy()
    after_missing = X_train.shape[1]

    zero_rate = X_train.fillna(0).eq(0).mean(axis=0)
    keep_zero = zero_rate <= zero_thresh
    X_train = X_train.loc[:, keep_zero].copy()
    X_test = X_test.loc[:, keep_zero].copy()
    after_mostly_zero = X_train.shape[1]

    prevalence = X_train.fillna(0).ne(0).mean(axis=0)
    keep_prevalence = prevalence >= prevalence_thresh
    X_train = X_train.loc[:, keep_prevalence].copy()
    X_test = X_test.loc[:, keep_prevalence].copy()
    after_prevalence = X_train.shape[1]

    keep_non_constant = X_train.nunique(dropna=True) > 1
    X_train = X_train.loc[:, keep_non_constant].copy()
    X_test = X_test.loc[:, keep_non_constant].copy()
    after_non_constant = X_train.shape[1]

    X_train = drop_duplicate_columns(X_train)
    X_test = X_test.loc[:, X_train.columns].copy()
    after_dedup = X_train.shape[1]

    variances = X_train.var(axis=0, skipna=True).fillna(0.0)
    if len(variances) > 0:
        variance_cutoff = variances.quantile(variance_quantile)
        keep_variance = variances > variance_cutoff
        if keep_variance.any():
            X_train = X_train.loc[:, keep_variance].copy()
            X_test = X_test.loc[:, keep_variance].copy()
    after_nzv = X_train.shape[1]

    train_medians = X_train.median(axis=0).fillna(0.0)
    X_train = X_train.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    filtering_summary = {
        "start_features": int(start_features),
        "after_non_na": int(after_non_na),
        "after_missing": int(after_missing),
        "after_mostly_zero": int(after_mostly_zero),
        "after_prevalence": int(after_prevalence),
        "after_non_constant": int(after_non_constant),
        "after_dedup": int(after_dedup),
        "after_nzv": int(after_nzv),
    }
    return X_train, X_test, filtering_summary

def safe_univariate_score(series, y, use_kruskal=True):
    values = np.asarray(series, dtype=float)
    labels = np.asarray(y)
    groups = [values[labels == cls] for cls in sorted(pd.unique(labels))]
    groups = [g[np.isfinite(g)] for g in groups]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
    try:
        stat, _ = kruskal(*groups) if use_kruskal else f_oneway(*groups)
        return float(stat) if np.isfinite(stat) else 0.0
    except Exception:
        return 0.0

def select_top_supervised_features(X_train, y_train, top_k=5000, use_kruskal=True):
    if X_train.shape[1] == 0:
        return [], pd.DataFrame(columns=["feature", "anova_or_kruskal", "mutual_info", "combined_score"])

    anova_scores = [safe_univariate_score(X_train[col], y_train, use_kruskal=use_kruskal) for col in X_train.columns]
    mi_scores = mutual_info_classif(X_train, y_train, discrete_features=False, random_state=42)

    score_df = pd.DataFrame({
        "feature": X_train.columns,
        "anova_or_kruskal": anova_scores,
        "mutual_info": mi_scores,
    })
    score_df = score_df.replace([np.inf, -np.inf], 0).fillna(0)
    score_df["anova_rank"] = score_df["anova_or_kruskal"].rank(method="average", ascending=False)
    score_df["mi_rank"] = score_df["mutual_info"].rank(method="average", ascending=False)
    score_df["combined_score"] = -(score_df["anova_rank"] + score_df["mi_rank"])
    score_df = score_df.sort_values(["combined_score", "anova_or_kruskal", "mutual_info"], ascending=False).reset_index(drop=True)
    selected = score_df.head(min(top_k, len(score_df)))["feature"].tolist()
    return selected, score_df

xgb_params = {
    "objective": "multi:softprob",
    "num_class": n_classes,
    "eval_metric": "mlogloss",
    "max_depth": 3,
    "min_child_weight": 5,
    "subsample": 0.7,
    "colsample_bytree": 0.05,
    "colsample_bynode": 0.05,
    "reg_alpha": 1.0,
    "reg_lambda": 1.0,
    "learning_rate": 0.05,
    "n_estimators": 300,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

xgb_params

{'objective': 'multi:softprob',
 'num_class': 4,
 'eval_metric': 'mlogloss',
 'max_depth': 3,
 'min_child_weight': 5,
 'subsample': 0.7,
 'colsample_bytree': 0.05,
 'colsample_bynode': 0.05,
 'reg_alpha': 1.0,
 'reg_lambda': 1.0,
 'learning_rate': 0.05,
 'n_estimators': 300,
 'tree_method': 'hist',
 'random_state': 42,
 'n_jobs': -1}

In [12]:
X_preview, _, preview_summary = filter_low_information_features(X_all, X_all)
pd.DataFrame([preview_summary])

,start_features,after_non_na,after_missing,after_mostly_zero,after_prevalence,after_non_constant,after_dedup,after_nzv
0,77447,77447,77447,42000,34457,34457,32639,32312


In [13]:
X_preview, _, _ = filter_low_information_features(X_all, X_all)
preview_selected, preview_scores = select_top_supervised_features(X_preview, y_all, top_k=5000, use_kruskal=True)
print(f"Preview top features selected: {len(preview_selected)}")
display(preview_scores.head(25))

Preview top features selected: 5000


,feature,anova_or_kruskal,mutual_info,anova_rank,mi_rank,combined_score
0,ra__k__Bacteria.p__Firmicutes.c__Tissierellia....,236.845980,0.089425,1.0,3.0,-4.0
1,ra__k__Bacteria.p__Firmicutes.c__Bacilli.o__Ba...,233.467094,0.070364,2.0,8.0,-10.0
2,ra__k__Bacteria.p__Firmicutes.c__Clostridia.o_...,233.237218,0.072882,3.0,7.0,-10.0
3,ra__k__Bacteria.p__Fusobacteria.c__Fusobacteri...,192.159554,0.068906,4.0,9.0,-13.0
4,ra__k__Bacteria.p__Firmicutes.c__Erysipelotric...,153.706866,0.076620,8.0,5.0,-13.0
5,ra__k__Bacteria.p__Firmicutes.c__Negativicutes...,188.057455,0.061564,5.0,15.0,-20.0
6,pa__UNINTEGRATED.g__Parvimonas.s__Parvimonas_m...,174.342291,0.061364,6.0,16.0,-22.0
7,pc__PWY.7220..adenosine.deoxyribonucleotides.d...,150.966522,0.062873,10.0,12.0,-22.0
8,pa__UDPNAGSYN.PWY..UDP.N.acetyl.D.glucosamine....,131.790687,0.073571,17.0,6.0,-23.0
9,PMID,119.890519,0.154804,30.0,1.0,-31.0


In [17]:
studies = sorted(study_all.unique())
lodo_results = []
fold_reports = {}
feature_rankings = {}

for held_out_study in studies:
    train_mask = study_all != held_out_study
    test_mask = study_all == held_out_study

    X_train_raw = X_all.loc[train_mask].reset_index(drop=True)
    X_test_raw = X_all.loc[test_mask].reset_index(drop=True)
    y_train = y_all.loc[train_mask].reset_index(drop=True)
    y_test = y_all.loc[test_mask].reset_index(drop=True)

    unseen_classes = sorted(set(y_test) - set(y_train))
    if unseen_classes:
        print(f"Skipping {held_out_study}: test set contains classes not present in training: {unseen_classes}")
        continue

    X_train, X_test, filtering_summary = filter_low_information_features(X_train_raw, X_test_raw)
    selected_cols, score_df = select_top_supervised_features(X_train, y_train, top_k=5000, use_kruskal=True)
    X_train = X_train.loc[:, selected_cols].copy()
    X_test = X_test.loc[:, selected_cols].copy()
    feature_rankings[held_out_study] = score_df

    class_labels = sorted(y_train.unique())
    label_to_int = {label: idx for idx, label in enumerate(class_labels)}
    int_to_label = {idx: label for label, idx in label_to_int.items()}
    y_train_encoded = y_train.map(label_to_int).to_numpy()

    model_params = dict(xgb_params)
    model_params["num_class"] = len(class_labels)
    model = XGBClassifier(**model_params)
    model.fit(X_train, y_train_encoded)

    pred_encoded = model.predict(X_test)
    pred_labels = pd.Series(pred_encoded).map(int_to_label)
    accuracy = accuracy_score(y_test, pred_labels)

    fold_cm = pd.crosstab(
        y_test,
        pred_labels,
        rownames=["actual"],
        colnames=["predicted"],
        dropna=False,
    )
    fold_report = classification_report(y_test, pred_labels, output_dict=True, zero_division=0)
    fold_reports[held_out_study] = {
        "confusion_matrix": fold_cm,
        "classification_report": pd.DataFrame(fold_report).T,
    }

    lodo_results.append({
        "held_out_study": held_out_study,
        "n_train": int(train_mask.sum()),
        "n_test": int(test_mask.sum()),
        "n_features_for_model": int(len(selected_cols)),
        "accuracy": accuracy,
        **filtering_summary,
    })

    print(
        f"{held_out_study}: accuracy={accuracy:.4f}, "
        f"train={int(train_mask.sum())}, test={int(test_mask.sum())}, "
        f"after_nzv={filtering_summary['after_nzv']}, model_features={len(selected_cols)}"
    )

lodo_results_df = pd.DataFrame(lodo_results).sort_values("accuracy", ascending=False).reset_index(drop=True)
mean_accuracy = lodo_results_df["accuracy"].mean() if not lodo_results_df.empty else float("nan")
weighted_accuracy = np.average(lodo_results_df["accuracy"], weights=lodo_results_df["n_test"]) if not lodo_results_df.empty else float("nan")

print(f"\nMean LODO accuracy: {mean_accuracy:.4f}")
print(f"Weighted LODO accuracy: {weighted_accuracy:.4f}")
lodo_results_df

FengQ_2015: accuracy=0.5195, train=1636, test=154, after_nzv=31882, model_features=5000
GuptaA_2019: accuracy=0.7833, train=1730, test=60, after_nzv=31743, model_features=5000
HMP_2012: accuracy=0.7347, train=1643, test=147, after_nzv=32684, model_features=5000
HanniganGD_2017: accuracy=0.3580, train=1709, test=81, after_nzv=32245, model_features=5000
ThomasAM_2018a: accuracy=0.4750, train=1710, test=80, after_nzv=31831, model_features=5000
ThomasAM_2018b: accuracy=0.6441, train=1731, test=59, after_nzv=31999, model_features=5000
ThomasAM_2019_c: accuracy=0.8375, train=1710, test=80, after_nzv=31633, model_features=5000
VogtmannE_2016: accuracy=0.6442, train=1686, test=104, after_nzv=32318, model_features=5000
WirbelJ_2018: accuracy=0.6800, train=1665, test=125, after_nzv=32505, model_features=5000
Skipping YachidaS_2019: test set contains classes not present in training: ['Other']
YuJ_2015: accuracy=0.7266, train=1662, test=128, after_nzv=31902, model_features=5000
ZellerG_2014: accur

,held_out_study,n_train,n_test,n_features_for_model,accuracy,start_features,after_non_na,after_missing,after_mostly_zero,after_prevalence,after_non_constant,after_dedup,after_nzv
0,ThomasAM_2019_c,1710,80,5000,0.837500,77447,77447,77447,41414,33726,33726,31953,31633
1,GuptaA_2019,1730,60,5000,0.783333,77447,77447,77447,41627,33843,33843,32064,31743
2,HMP_2012,1643,147,5000,0.734694,77447,77447,77447,41635,34849,34849,33015,32684
3,YuJ_2015,1662,128,5000,0.726562,77447,77447,77447,41037,34017,34012,32225,31902
4,WirbelJ_2018,1665,125,5000,0.680000,77447,77447,77447,41358,34657,34657,32834,32505
5,VogtmannE_2016,1686,104,5000,0.644231,77447,77447,77447,41376,34452,34452,32645,32318
6,ThomasAM_2018b,1731,59,5000,0.644068,77447,77447,77447,41641,34121,34121,32323,31999
7,ZellerG_2014,1634,156,5000,0.544872,77447,77447,77447,40299,33911,33911,32132,31810
8,FengQ_2015,1636,154,5000,0.519481,77447,77447,77447,40673,33996,33995,32205,31882
9,ThomasAM_2018a,1710,80,5000,0.475000,77447,77447,77447,41447,33925,33925,32153,31831


In [18]:
study_to_inspect = lodo_results_df.iloc[0]["held_out_study"]
print(f"Top supervised features from training folds when holding out: {study_to_inspect}")
display(feature_rankings[study_to_inspect].head(25))

Top supervised features from training folds when holding out: ThomasAM_2019_c


,feature,anova_or_kruskal,mutual_info,anova_rank,mi_rank,combined_score
0,ra__k__Bacteria.p__Firmicutes.c__Clostridia.o_...,216.860162,0.086731,1.0,4.0,-5.0
1,ra__k__Bacteria.p__Firmicutes.c__Tissierellia....,213.834534,0.065606,2.0,8.0,-10.0
2,ra__k__Bacteria.p__Firmicutes.c__Bacilli.o__Ba...,206.391756,0.064204,3.0,11.0,-14.0
3,ra__k__Bacteria.p__Firmicutes.c__Erysipelotric...,146.230964,0.063361,8.0,12.0,-20.0
4,PMID,119.205309,0.176277,20.0,1.0,-21.0
5,age,111.889189,0.091210,30.0,3.0,-33.0
6,pc__UNINTEGRATED.g__Peptostreptococcus.s__Pept...,117.084804,0.064322,24.0,10.0,-34.0
7,ra__k__Bacteria.p__Firmicutes.c__Negativicutes...,178.390972,0.053751,5.0,36.0,-41.0
8,pa__PWY.6700..queuosine.biosynthesis.g__Parvim...,117.296759,0.060399,23.0,18.0,-41.0
9,pa__PWY.6588..pyruvate.fermentation.to.acetone,133.332775,0.053927,11.0,35.0,-46.0


In [ ]:
study_to_inspect = lodo_results_df.iloc[0]["held_out_study"]
print(f"Inspecting held-out study: {study_to_inspect}")
display(fold_reports[study_to_inspect]["confusion_matrix"])
display(fold_reports[study_to_inspect]["classification_report"])
